In [1]:
#Task 2
# Eğitimi hızlandır. Bir eğitim adımının süresini ve saniyede işlenen
# token sayısını ölç. Sonra videodaki adımları sırayla ekle: TF32, bfloat16
# (ya da float16), torch.compile, flash attention, vocab'ı 50.304'e
# yuvarlamak. Her adımdan sonra süreyi tabloya yaz. Donanımın bir adımı
# desteklemiyorsa o adımı atla ve nedenini yaz, en az iki adımı ölçmen
# yeterli. Senin donanımında hangi adım en çok kazandırdı? (1:22:18 - 2:14:55)

In [2]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
!ls -la input.txt

-rw-r--r-- 1 root root 1115394 Nov  1 11:06 input.txt


In [3]:
from torch.cuda import is_available
#imports
import tiktoken
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import time
torch.manual_seed(1337)
if torch.cuda.is_available():
  torch.cuda.manual_seed(1337)

torch.set_float32_matmul_precision('high')

In [4]:
!nvidia-smi

Sun Nov  1 11:33:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             51W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
class CausalSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        #batchwisely k q v projections for all heads
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        #output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1.0
        #regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        #mask
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
            .view(1, 1, config.block_size, config.block_size))


    def forward(self, x):
        B, T, C = x.size() #batch size, sequence length, embedding dims

        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim = 2)
        k = k.view(B,T, self.n_head, C // self.n_head).transpose(1,2) #B, T, nh, hs to B, nh, T, hs
        q = q.view(B,T, self.n_head, C // self.n_head).transpose(1,2) #B, T, nh, hs to B, nh, T, hs
        v = v.view(B,T, self.n_head, C // self.n_head).transpose(1,2) #B, T, nh, hs to B, nh, T, hs
        #attention
        att = (q @ k.transpose(-2,-1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        y = att @ v #(B, nh, T, T) x (B, nh, T, hs) = (B, nh, T, hs)
        #taking all head outputs side by side
        y = y.transpose(1,2).contiguous().view(B,T,C)
        #output projection
        y = self.c_proj(y)
        return y

In [6]:
class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate = "tanh") #nonlinearity
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1.0
    def forward(self,x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x


In [7]:
class Block(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self,x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [8]:
@dataclass
class GPTConfig:
    block_size: int= 1024
    vocab_size: int = 50257
    n_layer: int= 12
    n_head: int= 12
    n_embd: int= 768

class GPT(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
                    wte = nn.Embedding(config.vocab_size, config.n_embd), #token embds
                    wpe = nn.Embedding(config.block_size, config.n_embd), #positional embds
                    h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]), #hidden blocks
                    ln_f = nn.LayerNorm(config.n_embd),)) #final layernorm

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False) #final classifier

        #weight tying // sharing
        self.transformer.wte.weight = self.lm_head.weight

        #param init
        self.apply(self._init_weights)

    #initialization parameter setting
    def _init_weights(self, module):
        #linear layer, embd -> N(0,0.02)
        #bias -> 0
        #layernorm parameters already comes from pytorch
        std = 0.02

        if hasattr(module, "NANOGPT_SCALE_INIT"):
            std *= (2 * self.config.n_layer) ** -0.5
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean = 0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean = 0.0, std =std)

    def forward(self, idx, targets=None):
        # idx is of shape (B, T)
        B, T = idx.size()
        assert T <= self.config.block_size, f"Cannot forward sequence of length {T}, block size is only {self.config.block_size}"
        #first embeddings token and positional
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device) # shape (T)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (T, n_embd)
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (B, T, n_embd)
        x = tok_emb + pos_emb
        #going for blocks
        for block in self.transformer.h:
            x = block(x)
        #final layernorm and the classifier
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x) # (B, T, vocab_size)
        loss = None

        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),targets.view(-1))
        return logits, loss

    @classmethod
    def from_pretrained(cls, model_type):
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        # create a from-scratch initialized minGPT model
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPT(GPTConfig())
model.to(device)

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (h): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=768, out_features=2304, bias=True)
          (c_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): GELU(approximate='tanh')
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [10]:
#how to load data to model
class DataLoaderLite:
    def __init__(self, B, T):
        self.B = B
        self.T = T
    #tokenizing them
        with open("input.txt", "r") as f:
            text = f.read()

        enc = tiktoken.get_encoding("gpt2")  #gpt2's encoder
        tokens = enc.encode(text)            #encoding data
        self.tokens = torch.tensor(tokens)

        print(f"loaded {len(self.tokens)} tokens")
        print(f"1 epoch = {len(self.tokens) // (B*T)} batches")

    #state
        self.current_position = 0

    def next_batch(self):
        B, T = self.B, self.T
        #as usual. gather B*T + 1 data. first B * T to input. shift input with one data point and make them target
        buf = self.tokens[self.current_position: self.current_position + (B * T) + 1] #taking batch + 1
        x = (buf[:-1]).view(B,T) #till last one
        y = (buf[1:]).view(B,T)  #second 2 last
        #advancing position in big token tensor
        self.current_position += B*T
        #prevention before hitting last data piece
        if self.current_position + (B * T + 1) > len(self.tokens):
            self.current_position = 0

        return x,y

In [11]:
train_loader = DataLoaderLite(16,1024)

loaded 338025 tokens
1 epoch = 20 batches


In [13]:
#optimizing loop
optimizer = torch.optim.AdamW(model.parameters(), lr = 3e-4)

for i in range(50):
    t0 = time.time()
    x,y = train_loader.next_batch()
    x,y = x.to(device), y.to(device)
    optimizer.zero_grad()
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
      logits, loss = model(x,y)
    loss.backward()
    optimizer.step()
    torch.cuda.synchronize()
    t1 = time.time()
    dt = (t1-t0) * 1000
    tokens_per_sec = (train_loader.B * train_loader. T) / (t1 - t0)
    print(f"step {i}, loss: {loss.item()}, dt: {dt:.2f}ms, tok/sec: {tokens_per_sec} ")


step 0, loss: 10.914588928222656, dt: 782.92ms, tok/sec: 20926.891972319772 
step 1, loss: 9.553516387939453, dt: 330.12ms, tok/sec: 49630.065949557786 
step 2, loss: 9.010405540466309, dt: 330.09ms, tok/sec: 49634.47508614955 
step 3, loss: 8.684435844421387, dt: 330.36ms, tok/sec: 49594.21257113165 
step 4, loss: 8.625044822692871, dt: 330.29ms, tok/sec: 49605.561131812334 
step 5, loss: 8.483414649963379, dt: 330.07ms, tok/sec: 49638.3830245839 
step 6, loss: 8.251243591308594, dt: 330.30ms, tok/sec: 49603.48435081884 
step 7, loss: 7.9690022468566895, dt: 330.35ms, tok/sec: 49595.64427989419 
step 8, loss: 7.7339277267456055, dt: 330.24ms, tok/sec: 49611.8641525562 
step 9, loss: 7.537411212921143, dt: 330.30ms, tok/sec: 49603.41274078266 
step 10, loss: 7.398852348327637, dt: 330.17ms, tok/sec: 49623.292461202625 
step 11, loss: 7.198834419250488, dt: 330.40ms, tok/sec: 49589.02331971886 
step 12, loss: 7.119551181793213, dt: 330.35ms, tok/sec: 49596.10960304046 
step 13, loss: 7.